In [22]:
# !nvidia-smi

In [23]:
# !sudo kill -9 38789 39513 39700 38789

In [24]:
import json
%matplotlib inline
import torch
import torch.nn as nn
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import re
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import string
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.metrics import mean_squared_error, confusion_matrix
from sklearn.model_selection import train_test_split
from helper import classToLabels, encode_sentence, pre_process_stem_sentence_array, correct_labels
from sklearn.metrics import classification_report
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import WordNetLemmatizer, PorterStemmer

from nltk.corpus import stopwords
import pandas as pd
from helper import nlp_pre_process_stem
from sklearn.model_selection import KFold, StratifiedKFold
from tqdm.notebook import tqdm
import os
from modelsAndDatasets import LSTMNetLatest, LSTMNet, DocumentDataset

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")


In [25]:
batch_size = 100

def validation_metrics(model, valid_dl):
    model.eval()
    correct = 0
    total = 0
    sum_loss = 0.0
    sum_rmse = 0.0
    for x, y, l in valid_dl:
        x = x.long().to(device)
        y = y.long().to(device)
        y_hat = model(x, l)
        loss = F.cross_entropy(y_hat, y, label_smoothing=0.1)
        pred = torch.max(y_hat, 1)[1]
        correct += (pred == y).float().sum()
        total += y.shape[0]
        sum_loss += loss.item() * y.shape[0]
        sum_rmse += np.sqrt(mean_squared_error(pred.cpu(), y.cpu().unsqueeze(-1))) * y.cpu().shape[0]
    return sum_loss / total, correct / total, sum_rmse / total


def load_dataLoader(df, train_index, val_index, X='encoded', y='label'):
    train_df = df.iloc[train_index, :]
    val_df = df.iloc[val_index, :]
#    train_df.to_excel(f'classification_folder_focal//training_split_{counter}.xlsx')
#    val_df.to_excel(f'classification_folder_focal//validation_split_{counter}.xlsx')

    X_train, y_train, X_valid, y_valid = list(train_df[X]), list(train_df[y]), list(
        val_df[X]), list(val_df[y])

    train_ds = DocumentDataset(X_train, y_train)
    valid_ds = DocumentDataset(X_valid, y_valid)
    # test_ds = DocumentDataset(X_test, y_test)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(valid_ds, batch_size=batch_size, shuffle=True)
    # test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=True)
    return train_dl, val_dl


def reset_weights(m):
    for layer in m.children():
        if hasattr(layer, 'reset_parameters'):
            print(f'Reset trainable parameters of layer = {layer}')
            layer.reset_parameters()


In [26]:
import os
import torch
from torch import nn
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import transforms
from sklearn.model_selection import KFold

results = {}

def train_model_cv(model, X, y, model_name='', cv=True, k=10, skip=0, epochs=10, lr=0.001, optimizer_state=None):
    # kf = KFold(k, shuffle=True, random_state=42)
    counter = 0
    df = pd.DataFrame()
    list_of_tuples = list(zip(X, y))

    skf = StratifiedKFold(k, shuffle=True, random_state=42)  
    df = pd.DataFrame(list_of_tuples, columns = ['encoded', 'label'])
    
    for train_index, val_index in skf.split(df['encoded'], df['label']):
        counter = counter + 1

        if skip > 0:
            skip -= 1
            continue

        min_val_loss = 1
        max_val_accuracy = -1

        train_dl, val_dl = load_dataLoader(df, train_index, val_index)

        model.apply(reset_weights)
        parameters = filter(lambda p: p.requires_grad, model.parameters())
        optimizer = torch.optim.Adam(parameters, lr=lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=4, factor=0.1, verbose=True)
        for i in range(epochs):
            print(f'Epoch: {i}')

            model.train()
            sum_loss = 0.0
            total = 0
            for inputs, target, l in tqdm(train_dl):
                inputs = inputs.long().to(device)
                target = target.long().to(device)
                y_pred = model(inputs, l)
                optimizer.zero_grad()
                loss = F.cross_entropy(y_pred, target, label_smoothing=0.1)
                loss.backward()
                optimizer.step()
                sum_loss += loss.item() * target.shape[0]
                total += target.shape[0]

            val_loss, val_acc, val_rmse = validation_metrics(model, val_dl)
            scheduler.step(val_loss)
            
#             print("epoch: %f/%f ->> train loss %f, val loss %f, val accuracy %f, and val rmse %.3f" % ((i + 1), epochs*k, sum_loss / total, val_loss, val_acc, val_rmse))
            if (val_loss < min_val_loss) & (val_acc.item() > max_val_accuracy):
                min_val_loss = val_loss
                max_val_accuracy = val_acc.item()
                print('saving model !!!')
                if model_name != '':
                    torch.save({'model_state_dict': model.state_dict()}, f'classification_folder//classificationModel_{model_name}_ForSplit-{counter}.pth')
                else:
                    torch.save({'model_state_dict': model.state_dict()}, f'classification_folder//classificationModel_ForSplit-{counter}.pth')

        # Evaluation for this fold
        correct, total = 0, 0
        y_pred = []
        y_true = []
        with torch.no_grad():
          # Iterate over the test data and generate predictions
            for inputs, targets, l in val_dl:
                inputs = inputs.long().to(device)
                targets = targets.long().to(device)
                
                outputs = model(inputs, l)
                _, predicted = torch.max(outputs.data, 1)
                total += targets.size(0)
                correct += (predicted == targets).sum().item()

                for out in predicted.cpu():
                    y_pred.append(out)
                for out in targets.cpu():
                    y_true.append(out)

          # Print accuracy
            print('Accuracy for fold %d: %d %%' % (counter, 100.0 * correct / total))
            print('--------------------------------')
            results[counter] = (100.0 * (correct / total), classification_report(y_true, y_pred))

        # Print fold results
        print(f'K-FOLD CROSS VALIDATION RESULTS FOR {k} FOLDS')
        print('--------------------------------')
        total_sum = 0.0
        for key, value in results.items():
            print(f'Fold {key}: {value[0]} %')
            print(value[1])
            total_sum += value[0]
        print(f'Average: {total_sum/len(results.items())} %')
        if not cv:
            break

#             if i == (epochs - 1):
#                 print('saving last model -- last_checkpoint_LR' + str(i + 1))
#                 torch.save({
#                     'epoch': i + 1,
#                     'model_state_dict': model.state_dict(),
#                     'optimizer_state_dict': optimizer.state_dict(),
#                     'train_loss': sum_loss / total,
#                     'val_loss': val_loss
#                 }, 'last_checkpoint_10-4' + str(i + 1) + '.pth')


In [27]:
data = pd.read_excel('doc_classification_training_base.xlsx')
data = data[data.text.str.contains(' ', na=False)]
data['text_length'] = data['text'].apply(lambda x: len(x.split()))

# Zero-numbering the labels
# data['DOCUMENT_TYPE'] = classToLabels(data['DOCUMENT_TYPE'])
mean_words = int(np.ceil(np.mean(data['text_length'])))
# mean_words = 100
print(mean_words)

counts = Counter()
for index, row in data.iterrows():
    counts.update(row['text'].split())

# deleting infrequent words
# print("num_words before:", len(counts.keys()))
# for word in list(counts):
#     if counts[word] < 5:
#         del counts[word]
# print("num_words after:", len(counts.keys()))

# creating vocabulary
vocab2index = {"": 0, "UNK": 1}
words = ["", "UNK"]
for word in counts:
    vocab2index[word] = len(words)
    words.append(word)

data['encoded'] = data['text'].apply(lambda x: np.array(encode_sentence(x, vocab2index, mean_words)))
# data1 = data[data['DOCUMENT_TYPE']]

X = list(data['encoded'])
y = list(data['DOCUMENT_TYPE'])

187


In [28]:
embedding_dim = 100
hidden_dim = 100
n_classes = 10
vocab_size = len(words)
model_name='baseData'
# optimizer_state = torch.load('last_checkpoint_100.pth')['optimizer_state_dict']

model_fixed = LSTMNet(vocab_size, embedding_dim=embedding_dim, hidden_dim=hidden_dim, n_classes=n_classes)
model_fixed.to(device)
# model_fixed.load_state_dict(torch.load('best_checkpoint_cv.pth')['model_state_dict'])
# print(model_fixed)

train_model_cv(model_fixed, X, y, model_name=model_name, k=5, epochs=50, lr=0.001)

Reset trainable parameters of layer = Embedding(115886, 100, padding_idx=0)
Reset trainable parameters of layer = LSTM(100, 100, batch_first=True, bidirectional=True)
Reset trainable parameters of layer = Linear(in_features=100, out_features=10, bias=True)
Epoch: 0


  0%|          | 0/232 [00:00<?, ?it/s]

saving model !!!
Epoch: 1


  0%|          | 0/232 [00:00<?, ?it/s]

saving model !!!
Epoch: 2


...

Epoch: 48


  0%|          | 0/232 [00:00<?, ?it/s]

Epoch: 49


  0%|          | 0/232 [00:00<?, ?it/s]

Accuracy for fold 5: 93 %
--------------------------------
K-FOLD CROSS VALIDATION RESULTS FOR 5 FOLDS
--------------------------------
Fold 1: 93.73922042083477 %
              precision    recall  f1-score   support

           0       0.86      0.80      0.83       387
           1       0.94      0.94      0.94      1728
           2       0.92      0.96      0.94       588
           3       0.92      0.93      0.93      1538
           4       0.98      0.98      0.98       855
           5       0.96      0.92      0.94       250
           6       0.98      0.98      0.98       422
           7       0.95      0.75      0.84        24
           8       1.00      1.00      1.00         6

    accuracy                           0.94      5798
   macro avg       0.95      0.92      0.93      5798
weighted avg       0.94      0.94      0.94      5798

Fold 2: 94.1521476625841 %
              precision    recall  f1-score   support

           0       0.85      0.81      0.83      

In [29]:
# train_model(model_fixed, X, y, epochs=10, lr=0.0001)

In [30]:
final_obj = {'vocab2index': vocab2index, 'mean_words': mean_words,
             'vocab_size': vocab_size, 'embedding_dim': embedding_dim,
             'hidden_dim': hidden_dim, 'n_classes': n_classes}
json.dump(final_obj, open(f'vector_v1_{model_name}.txt', 'w'))

In [31]:
json.dump(results, open(f'results_{model_name}.txt', 'w'))

In [32]:
# X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42, stratify=y)

# # Separate test and validation.
# # X_test, X_valid, y_test, y_valid = train_test_split(X_valid, y_valid, test_size=0.5, shuffle=True, random_state=42, stratify=y_valid)

# train_ds = DocumentDataset(X_train, y_train)
# valid_ds = DocumentDataset(X_valid, y_valid)
# # test_ds = DocumentDataset(X_test, y_test)

# batch_size = 500
# vocab_size = len(words)
# train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
# val_dl = DataLoader(valid_ds, batch_size=batch_size, shuffle=True)
# # test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=True)

# embedding_dim = 100
# hidden_dim = 100
# n_classes = 8
# # optimizer_state = torch.load('last_checkpoint_100.pth')['optimizer_state_dict']

# model_fixed = LSTM_fixed_len(vocab_size, embedding_dim=embedding_dim, hidden_dim=hidden_dim, n_classes=n_classes)
# # model_fixed.load_state_dict(torch.load('last_checkpoint_100.pth')['model_state_dict'])
# print(model_fixed)


In [33]:
# torch.rand(10).numpy()

In [34]:
# # Metrics
# y_true = []
# y_pred = []
# with torch.no_grad():
#     for inputs, classes, i in val_dl:
#         inputs = inputs.long()
#         classes = classes.long()
#         temp = model_fixed(inputs, 1)
#         _, outputs = torch.max(temp, 1)
#         for out in outputs:
#             y_pred.append(out)
#         for out in classes:
#             y_true.append(out)

# metrics = classification_report(y_true, y_pred)
# print(metrics)

In [35]:
# train_model(model_fixed, train_dl, val_dl, epochs=200, lr=0.00001)